In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from Bio.Seq import Seq
from evo.dataset import CherriesDataset, EncodedPEINTDataset
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm

tqdm.pandas()

from plmr.data.datamodule import PLMRDataModule
from plmr.models.frameworks.peint import load_from_new_checkpoint

In [4]:
# Load trained models from checkpoints
ckpt_dir = Path("/accounts/projects/yss/stephen.lu/protevo/plmr/logs/train/runs")
ckpt_paths = {
    "heavy": ckpt_dir / "2025-08-08_15-36-04/checkpoints/epoch_089.ckpt",
    "light": ckpt_dir / "2025-08-10_11-57-18/checkpoints/epoch_042.ckpt",
    "joint": ckpt_dir / "2025-08-10_11-58-46/checkpoints/epoch_044.ckpt",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

modules = {k: load_from_new_checkpoint(v, device) for k, v in ckpt_paths.items()}
vocab = next(iter(modules.values())).net.vocab

Using device: cuda


/accounts/projects/yss/stephen.lu/protevo/plmr/plmr/models/frameworks/peint.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map

In [7]:
# Calculate likelihood of leaves conditioned on the root and distance from root
import tempfile

def dataloader_from_transitions(transitions, batch_size=32, mask_prob=0.0):
    datafile = tempfile.NamedTemporaryFile(delete=False, suffix=".txt")
    with open(datafile.name, "w") as f:
        f.write("{0} transitions\n".format(len(transitions)))
        f.write("\n".join(transitions))

    dataset = EncodedPEINTDataset(
        dataset=CherriesDataset(data_file=datafile.name),
        vocab=vocab,
        mask_prob=mask_prob,
        random_token_prob=0.0,
        leave_unmasked_prob=0.0,
    )
    generator = iter(
        PLMRDataModule(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=False,
        )._dataloader_template(dataset=dataset, training=False)
    )
    return generator

def infer_log_likelihoods(dataloader, module):
    # run inference on the dataloader
    lls = []
    for batch in tqdm(dataloader, desc="Inference"):
        batch = [b.to(device) for b in batch]
        [x, x_targets, y, y_targets, t, x_pad_mask, y_pad_mask] = batch
        yt_mask = y_targets != module.net.vocab.pad_idx  # actual values

        with torch.no_grad() and torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            x_logits, y_logits = module(x, y, t, x_pad_mask, y_pad_mask)

        y_logits = y_logits - torch.logsumexp(y_logits, dim=-1, keepdim=True)
        y_logits = y_logits.transpose(-1, -2)
        nll = F.cross_entropy(y_logits, y_targets, ignore_index=vocab.pad_idx, reduction="none")

        ll = -nll * yt_mask.float()
        ll = ll.sum(dim=-1)
        lls.append(ll.detach().cpu().numpy())

    lls = np.concatenate(lls)
    return lls

In [8]:
df = pd.read_csv("/scratch/users/stephen.lu/projects/protevo/data/flab/Koenig2017_g6_er.csv")
df.head()

,heavy,light,fitness
0,EAQLVESGGGLVQPGGSLRLSCAASGFTISDYWIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDVSTAVAWYQQKPGKAPKL...,0.819679
1,ECQLVESGGGLVQPGGSLRLSCAASGFTISDYWIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDVSTAVAWYQQKPGKAPKL...,0.328397
2,EDQLVESGGGLVQPGGSLRLSCAASGFTISDYWIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDVSTAVAWYQQKPGKAPKL...,1.585834
3,EEQLVESGGGLVQPGGSLRLSCAASGFTISDYWIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDVSTAVAWYQQKPGKAPKL...,1.806183
4,EFQLVESGGGLVQPGGSLRLSCAASGFTISDYWIHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDVSTAVAWYQQKPGKAPKL...,1.452152


In [14]:
heavy_wt = "EVQLVESGGGLVQPGGSLRLSCAASGFTISDYWIHWVRQAPGKGLEWVAGITPAGGYTYYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARFVFFLPYAMDYWGQGTLVTVSS"
light_wt = "DIQMTQSPSSLSASVGDRVTITCRASQDVSTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQSYTTPPTFGQGTKVEIKR"
print(len(heavy_wt), len(light_wt), len(heavy_wt) + len(light_wt))

120 108 228


In [ ]:
heavy_transitions = [
    f"{heavy_wt.strip()} {row.heavy.strip()} 7.0" for _, row in df.iterrows()
]
dataloader = dataloader_from_transitions(heavy_transitions, batch_size=32)
heavy_lls = infer_log_likelihoods(dataloader, modules["heavy"])

Inference: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 134/134 [00:21<00:00,  6.34it/s]


In [12]:
light_transitions = [
    f"{light_wt.strip()} {row.light.strip()} 7.0" for _, row in df.iterrows()
]
dataloader = dataloader_from_transitions(light_transitions, batch_size=32)
light_lls = infer_log_likelihoods(dataloader, modules["light"])

Inference: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 134/134 [00:10<00:00, 12.49it/s]


In [15]:
df['heavy_ll'] = heavy_lls
df['light_ll'] = light_lls
df['joint_ll'] = df['heavy_ll'] + df['light_ll']
df['ppl'] = np.exp(-df['joint_ll'] / (len(heavy_wt) + len(light_wt)))

In [16]:
df.to_csv("/scratch/users/stephen.lu/projects/protevo/data/flab/Koenig2017_g6_peint_lls_t=7.csv", index=False)